# Step 2 — One provisional reference award per identified author

Read the frozen best-paper cohort without modifying it. Build one row per resolved OpenAlex author ID, anchored to that author's **earliest identified award year within the frozen 2000–2018 cohort**.

This is not necessarily the person's first-ever award. Missing authorships and IDs can hide earlier awards. The output is explicitly provisional until those gaps have been addressed.

When multiple awards occur in the earliest year, retain ALL earliest-year award records in a linked table and JSON column. Select one deterministic representative paper only for convenience; do not treat its position/institution as uniquely representing all awards in that year.

No API calls, name-based identity merging, seniority recalculation, publication/citation calculations, or control-data processing. Suggested placement: `notebooks/02_reference_awards.ipynb`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil
import tempfile
import unicodedata
import pandas as pd
from IPython.display import display

ROOT_OVERRIDE = None
FREEZE_REL = 'conf_paper/conf_data/frozen/award_cohort_v1'
OUTPUT_VERSION = 'reference_awards_v1'
WRITE_OUTPUTS = True

def find_root():
    if ROOT_OVERRIDE:
        root = Path(ROOT_OVERRIDE).expanduser().resolve()
        if not (root / FREEZE_REL / 'manifest.json').exists():
            raise FileNotFoundError('Configured root has no frozen cohort manifest.')
        return root
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / FREEZE_REL / 'manifest.json').exists():
            return p
    raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def strict_bool(s):
    values = s.fillna('').astype(str).str.strip().str.lower()
    unknown = set(values) - {'true', 'false', ''}
    if unknown:
        raise ValueError('Unexpected boolean values: ' + repr(unknown))
    return values.eq('true')

def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

ROOT = find_root()
FROZEN = ROOT / FREEZE_REL
TARGET = ROOT / 'conf_paper' / 'conf_data' / 'derived' / OUTPUT_VERSION
manifest = json.loads((FROZEN / 'manifest.json').read_text(encoding='utf-8'))
for name, expected_hash in manifest['output_sha256'].items():
    p = FROZEN / name
    if not p.is_file() or digest(p) != expected_hash:
        raise RuntimeError(f'Frozen integrity check failed: {name}. Do not bypass; investigate edits or Git line-ending conversion.')
papers = read_csv(FROZEN / 'award_cohort_papers.csv')
authors = read_csv(FROZEN / 'award_cohort_authorships.csv')
assert papers.cohort_record_id.is_unique
assert authors.authorship_record_id.is_unique
assert set(authors.cohort_record_id) == set(papers.cohort_record_id)
papers['award_year'] = pd.to_numeric(papers.award_year, errors='raise').astype(int)
authors['award_year'] = pd.to_numeric(authors.award_year, errors='raise').astype(int)
for col in ['paper_has_recorded_openalex_match', 'paper_authorships_unavailable', 'paper_author_data_incomplete']:
    papers[col] = strict_bool(papers[col])
for col in ['is_placeholder', 'author_identity_unresolved']:
    authors[col] = strict_bool(authors[col])
print('Frozen input verified:', manifest['cohort_version'])

Frozen input verified: award_cohort_v1


## 1. Inspect the newly identified gaps

Count matched papers with unavailable author lists separately from unmatched papers. Show the conference distribution before selecting references; no paper is removed from the frozen cohort.

In [2]:
gap_breakdown = papers.groupby(['paper_has_recorded_openalex_match', 'paper_authorships_unavailable'], dropna=False).size().rename('award_records').reset_index()
display(gap_breakdown)
conference_gaps = papers.groupby('conference', dropna=False).agg(
    award_records=('cohort_record_id', 'size'),
    matched_work_id=('paper_has_recorded_openalex_match', 'sum'),
    unavailable_authorship_lists=('paper_authorships_unavailable', 'sum'),
    incomplete_author_data=('paper_author_data_incomplete', 'sum')).reset_index()
display(conference_gaps.loc[conference_gaps.incomplete_author_data.gt(0)])
unavailable_papers = papers.loc[papers.paper_authorships_unavailable].copy()
review_cols = [c for c in ['cohort_record_id', 'conference', 'award_year', 'paper_title',
                         'openalex_work_id', 'authorship_parse_status', 'source_authors'] if c in unavailable_papers]
display(unavailable_papers[review_cols])

,paper_has_recorded_openalex_match,paper_authorships_unavailable,award_records
0,False,True,22
1,True,False,868
2,True,True,35


,conference,award_records,matched_work_id,unavailable_authorship_lists,incomplete_author_data
0,AAAI,26,26,0,1
2,CHI,192,185,7,8
5,FOCS,31,30,2,4
6,FSE,63,63,0,2
10,ICWSM,15,15,15,15
11,IJCAI,27,25,2,2
12,INFOCOM,24,24,0,1
13,JCDL,20,19,20,20
15,MOBICOM,11,10,1,1
17,OSDI,21,19,2,2


,cohort_record_id,conference,award_year,paper_title,openalex_work_id,authorship_parse_status,source_authors
151,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,SODA,2017,A (2 + Є) Approximation for Maximum Weight Mat...,,missing,"Ami Paz, Institut de Recherche en Informatique..."
213,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,OSDI,2016,Ryoan: A Distributed Sandbox for Untrusted Com...,,missing,"Tyler Hunt, University of Texas at Austin ; et..."
214,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,OSDI,2016,Early Detection of Configuration Errors to Red...,,missing,"Tianyin Xu, University of California, San Dieg..."
261,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2015,"""I always assumed that I wasn't really that cl...",,missing,"Motahhare Eslami, University of Illinois at Ur..."
333,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,FOCS,2014,Path Finding Methods for Linear Programming: S...,W4240592614,empty_list,"Yin Tat Lee & Aaron Sidford, Massachusetts Ins..."
398,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2013,At Home with Agents: Exploring Attitudes Towar...,,missing,"Tom Rodden, University of Nottingham ; et al. ..."
416,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,IJCAI,2013,Bayesian Optimization in High Dimensions via R...,,missing,"Ziyu Wang, University of British Columbia ; et..."
451,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2012,Communitysourcing: Engaging Local Crowds to Pe...,,missing,"Kurtis Heimerl, University of California, Berk..."
452,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2012,"Touché: Enhancing Touch Interaction on Humans,...",,missing,"Munehiko Sato, Disney Research ; et al. Ivan P..."
461,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,CHI,2012,ClayVision: The (Elastic) Image of the City,,missing,"Yuichiro Takeuchi, Sony Ken Perlin, New York U..."


## 2. Keep unresolved identities separate

Missing-ID slots and placeholder rows remain in an unresolved queue. They are not guessed to be distinct authors and are not merged with identified authors on name alone. A valid ID's existence does not itself validate OpenAlex disambiguation.

In [3]:
well_formed_id = authors.author_id.str.fullmatch(r'A\d+')
resolved_mask = ~authors.is_placeholder & ~authors.author_identity_unresolved & well_formed_id
resolved = authors.loc[resolved_mask].copy()
unresolved = authors.loc[~resolved_mask].copy()
if resolved.empty:
    raise ValueError('No resolved author IDs available for reference selection.')
assert not resolved.author_id.eq('').any()
events = resolved.drop_duplicates(['author_id', 'cohort_record_id']).copy()
events = events.merge(papers[['cohort_record_id', 'paper_title']], on='cohort_record_id',
                      how='left', validate='many_to_one')
print('Identified authors:', events.author_id.nunique())
print('Identified author-award records:', len(events))
print('Unresolved observed slots:', int((~unresolved.is_placeholder).sum()))
print('Unknown-list placeholders (not authors):', int(unresolved.is_placeholder.sum()))

Identified authors: 2710
Identified author-award records: 3150
Unresolved observed slots: 52
Unknown-list placeholders (not authors): 57


## 3. Select earliest identified reference year; retain same-year ties

Use the earliest available award year per ID. The convenience representative is sorted by conference, work ID, then frozen record ID; this tie-break does not imply that one award preceded another within the year. Names are not taken as identity keys.

In [4]:
first_year = events.groupby('author_id').award_year.min().rename('reference_award_year')
events = events.merge(first_year, on='author_id', validate='many_to_one')
events['is_reference_year'] = events.award_year.eq(events.reference_award_year)
reference_events = events.loc[events.is_reference_year].copy()
reference_events = reference_events.sort_values(['author_id', 'conference', 'openalex_work_id', 'cohort_record_id'], kind='stable')
representatives = reference_events.drop_duplicates('author_id', keep='first').copy()
any_cohort_gaps = bool(papers.paper_author_data_incomplete.any())

roster_rows = []
for aid, group in events.groupby('author_id', sort=True):
    earliest = reference_events.loc[reference_events.author_id.eq(aid)]
    representative = representatives.loc[representatives.author_id.eq(aid)].iloc[0]
    ref_year = int(group.reference_award_year.iloc[0])
    names = sorted({x.strip() for x in group.author_name if x.strip()})
    tied = len(earliest) > 1
    fields = [c for c in ['cohort_record_id', 'openalex_work_id', 'conference', 'paper_title',
                          'author_order_in_source', 'author_position', 'institutions_json'] if c in earliest]
    roster_rows.append({
        'author_id': aid, 'author_name': representative.author_name,
        'observed_name_variants_json': json.dumps(names, ensure_ascii=False),
        'reference_award_year': ref_year,
        'reference_rule': 'earliest_identified_award_year_in_frozen_2000_2018_cohort',
        'reference_selection_provisional': any_cohort_gaps,
        'n_award_records_in_cohort': int(group.cohort_record_id.nunique()),
        'n_award_years_in_cohort': int(group.award_year.nunique()),
        'is_repeat_awardee_in_identified_data': group.cohort_record_id.nunique() > 1,
        'latest_identified_award_year': int(group.award_year.max()),
        'n_awards_in_reference_year': len(earliest),
        'reference_year_has_multiple_awards': tied,
        'representative_cohort_record_id': representative.cohort_record_id,
        'representative_work_id': representative.openalex_work_id,
        'representative_conference': representative.conference,
        'representative_paper_title': representative.paper_title,
        'representative_author_order': representative.author_order_in_source,
        'representative_author_position': representative.author_position,
        'representative_institutions_json': representative.get('institutions_json', ''),
        'reference_year_award_records_json': json.dumps(earliest[fields].to_dict('records'), ensure_ascii=False)
    })
roster = pd.DataFrame(roster_rows)
assert roster.author_id.is_unique
assert len(roster) == resolved.author_id.nunique()
assert roster.reference_award_year.between(2000, 2018).all()
display(roster[['author_id', 'author_name', 'reference_award_year', 'n_award_records_in_cohort',
                'n_awards_in_reference_year', 'reference_selection_provisional']].head(20))

,author_id,author_name,reference_award_year,n_award_records_in_cohort,n_awards_in_reference_year,reference_selection_provisional
0,A5000009841,Ana Klimovic,2014,1,1,True
1,A5000102893,David Akers,2009,1,1,True
2,A5000107991,Michael Vrable,2008,1,1,True
3,A5000163896,Elisa D. Mekler,2016,1,1,True
4,A5000167011,I. Scott MacKenzie,2008,1,1,True
5,A5000170074,Alexandru Agache,2017,1,1,True
6,A5000245150,Alexander J. Smola,2014,1,1,True
7,A5000246567,Matthew L. Lee,2014,1,1,True
8,A5000294737,Philipp Schwartenbeck,2016,1,1,True
9,A5000308912,Jaechang Nam,2013,1,1,True


## 4. Flag possible earlier awards hidden by missing IDs

Exact normalized name agreement is only a review signal: different people can share names and the same person can use different names. This never changes reference years or merges identities. Unknown-list papers cannot be checked this way; this limitation is flagged for the entire roster.

In [5]:
def name_key(s):
    return ' '.join(unicodedata.normalize('NFKC', str(s)).casefold().split())

name_to_ids = {}
for row in events[['author_id', 'author_name']].itertuples(index=False):
    key = name_key(row.author_name)
    if key:
        name_to_ids.setdefault(key, set()).add(row.author_id)
ref_lookup = roster.set_index('author_id').reference_award_year.to_dict()
suspect_rows = []
for row in unresolved.loc[~unresolved.is_placeholder].itertuples(index=False):
    for aid in name_to_ids.get(name_key(row.author_name), set()):
        if row.award_year <= ref_lookup[aid]:
            suspect_rows.append({'candidate_author_id': aid, 'unresolved_authorship_record_id': row.authorship_record_id,
                                 'unresolved_name': row.author_name, 'unresolved_award_year': row.award_year,
                                 'current_reference_year': ref_lookup[aid],
                                 'reason': 'exact_normalized_name_only; manual_review_required'})
name_review = pd.DataFrame(suspect_rows, columns=['candidate_author_id', 'unresolved_authorship_record_id',
                       'unresolved_name', 'unresolved_award_year', 'current_reference_year', 'reason'])
roster['possible_earlier_or_same_year_unresolved_name_match'] = roster.author_id.isin(name_review.candidate_author_id)
roster['unavailable_paper_authorships_exist_in_cohort'] = bool(papers.paper_authorships_unavailable.any())
summary = pd.DataFrame([
    ('resolved_unique_authors', len(roster)),
    ('identified_repeat_awardees', int(roster.is_repeat_awardee_in_identified_data.sum())),
    ('authors_with_multiple_reference_year_awards', int(roster.reference_year_has_multiple_awards.sum())),
    ('authors_flagged_for_unresolved_name_review', int(roster.possible_earlier_or_same_year_unresolved_name_match.sum())),
    ('unresolved_observed_author_slots', int((~unresolved.is_placeholder).sum())),
    ('unavailable_authorship_papers', len(unavailable_papers))
], columns=['measure', 'value'])
display(summary)
display(name_review)

,measure,value
0,resolved_unique_authors,2710
1,identified_repeat_awardees,308
2,authors_with_multiple_reference_year_awards,27
3,authors_flagged_for_unresolved_name_review,1
4,unresolved_observed_author_slots,52
5,unavailable_authorship_papers,57


,candidate_author_id,unresolved_authorship_record_id,unresolved_name,unresolved_award_year,current_reference_year,reason
0,A5098429303,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,Mary Pogue,2004,2004,exact_normalized_name_only; manual_review_requ...


## 5. Export provisional author roster and audit links

Default: write a separate derived directory, never edit the frozen cohort. A rerun verifies and reuses an existing identical result rather than overwriting it. If inputs or transform version change, choose a new output version after review.

In [6]:
transform_version = 'reference_awards_1.0'
input_hashes = {name: digest(FROZEN / name) for name in
                ['manifest.json', 'award_cohort_papers.csv', 'award_cohort_authorships.csv']}
identity = {'input_hashes': input_hashes, 'transform_version': transform_version,
            'reference_rule': 'earliest_identified_award_year_in_frozen_2000_2018_cohort',
            'tie_break': 'conference, work ID, cohort record ID; representative only'}

if WRITE_OUTPUTS:
    if TARGET.exists():
        prior = json.loads((TARGET / 'manifest.json').read_text(encoding='utf-8'))
        if prior['identity'] != identity:
            raise RuntimeError('Different existing inputs/settings. Review and choose a new OUTPUT_VERSION.')
        for name, expected in prior['output_sha256'].items():
            if not (TARGET / name).is_file() or digest(TARGET / name) != expected:
                raise RuntimeError('Existing derived output integrity failed: ' + name)
        print('Verified existing step-2 outputs; nothing overwritten:', TARGET)
    else:
        TARGET.parent.mkdir(parents=True, exist_ok=True)
        stage = Path(tempfile.mkdtemp(prefix='.' + OUTPUT_VERSION + '_', dir=TARGET.parent))
        try:
            tables = {'author_reference_roster.csv': roster, 'author_award_links.csv': events,
                      'reference_year_award_links.csv': reference_events,
                      'unresolved_authorship_queue.csv': unresolved,
                      'papers_without_authorships.csv': unavailable_papers,
                      'possible_earlier_unresolved_names.csv': name_review,
                      'conference_authorship_gaps.csv': conference_gaps,
                      'matching_authorship_gap_breakdown.csv': gap_breakdown,
                      'step2_summary.csv': summary}
            for name, frame in tables.items():
                frame.to_csv(stage / name, index=False)
            notes = ('# Provisional author reference awards\n\n'
                     'One row per resolved OpenAlex author ID. Reference year = earliest identified award year '
                     'within the frozen 2000–2018 cohort, NOT first-ever award. All same-year ties and all award '
                     'links are retained. No missing IDs were resolved by name. Unresolved slots and unknown-list '
                     'papers remain in audit queues and are not counted as known unique authors. '
                     'Recover missing identities/authorships before treating reference assignments as final. '
                     'The representative paper is a deterministic convenience only. Frozen source data are unchanged.\n')
            (stage / 'README.md').write_text(notes, encoding='utf-8')
            hashes = {p.name: digest(p) for p in sorted(stage.iterdir()) if p.is_file()}
            meta = {'created_at_utc': datetime.now(timezone.utc).isoformat(),
                    'freeze_version': manifest['cohort_version'], 'identity': identity,
                    'output_sha256': hashes, 'external_api_calls': False, 'frozen_source_modified': False}
            (stage / 'manifest.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
            if TARGET.exists():
                raise FileExistsError('Output directory appeared during export; refusing to overwrite.')
            stage.rename(TARGET)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
        print('Created provisional step-2 outputs:', TARGET)
else:
    print('Preview only. No outputs written.')

Created provisional step-2 outputs: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\derived\reference_awards_v1
